# 04 · Stock-out runs: negative binomial + Benjamini–Hochberg (PySpark)

**The test.** How likely is an item to sell nothing for k full trading days in a row, given its normal rate?
Tiny p means something was wrong on the shelf. Poisson (**p = e^(−λk)**) assumes variance = mean, but grocery
demand is far lumpier, so the test uses a **negative binomial**: mean λ and dispersion φ = variance ÷ mean, both
from the previous 28 trading days, giving **p = φ^(−λk/(φ−1))**. It's exactly Poisson when φ ≤ 1. `p_poisson`
is kept for comparison.

**Which runs are tested.** Runs that start in the analysis window, have 28 days of history, λ > 0, k ≥ 1, and end
with a sale within 28 trading days. Longer gaps look like delisting. A partial trading day (receipts < half
the store's normal) doesn't count as evidence.

**Multiple testing.** Millions of runs are tested, so a flat 5% cut-off would flag thousands by chance.
**Benjamini–Hochberg** picks the cut-off so that *of the runs we flag, at most 5% are expected to be chance*.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../src"))  # the repo's src/, via the Databricks Git folder

from favorita_spark.config import Config

cfg = Config()
spark.sql(f"USE CATALOG {cfg.catalog}")

In [ ]:
from pyspark.sql import functions as F

from favorita_spark.stockout import bh_summary, dispersion, flag_bh, stockout_runs

series = spark.table("silver.stockout_series")
stockout_runs(series, cfg).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.stockout_run_tested")

# serverless has no .cache(): rank the materialised table instead of recomputing the plan
runs, cutoff = flag_bh(spark.table("silver.stockout_run_tested"), cfg.fdr_q)
runs.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.fact_stockout_run")

In [ ]:
summary = bh_summary(spark.table("gold.fact_stockout_run"), cutoff)
spark.createDataFrame([summary], "tested_runs BIGINT, flagged_runs BIGINT, bh_cutoff DOUBLE, max_chance_flags DOUBLE") \
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.stockout_bh_summary")
print(f"Tested {summary['tested_runs']:,} runs; flagged {summary['flagged_runs']:,} at p ≤ {summary['bh_cutoff'] or 0:.2e}.")
print(f"At most {summary['max_chance_flags']:,.0f} of those flags would appear by chance alone.")

## Why not plain Poisson?

Poisson assumes variance = mean. Grocery demand is **over-dispersed** (variance > mean), which makes long zero
runs likelier than Poisson thinks. On a 3-store trial, Poisson flagged 38% of all runs. The next cells measure
the dispersion and compare how many runs each model would flag.

In [ ]:
dispersion(series, cfg).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.item_dispersion")
display(spark.sql("""
    SELECT percentile(dispersion, 0.5) AS median_dispersion, avg(CAST(dispersion > 1 AS INT)) AS share_over_1
    FROM gold.item_dispersion
"""))
poisson_flags, poisson_cutoff = flag_bh(spark.table("silver.stockout_run_tested").withColumn("p_value", F.col("p_poisson")), cfg.fdr_q)
print(f"Poisson would flag {poisson_flags.filter('is_flagged').count():,} runs; "
      f"the negative binomial flags {summary['flagged_runs']:,}.")